# Experiment 6: Stacking Ensemble for Delivery Time Prediction

This notebook implements a stacking ensemble combining LightGBM and Random Forest base models with a Linear Regression meta-learner.

**Pipeline:**
1. Load & preprocess data (Dataset A)
2. Load tuned base models from Experiments 4 & 5
3. Build StackingRegressor with Linear Regression meta-learner
4. Evaluate and compare against base models
5. Create deployment-ready predictor class

In [ ]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, json
warnings.filterwarnings('ignore')

from sklearn import set_config
set_config(transform_output='pandas')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import clean_data_utils

## 2. Load Data & Recreate Split (same as Exp 3/4/5)

In [ ]:
raw_df = pd.read_csv('food_delivery_data.csv')
clean_data_utils.perform_data_cleaning(raw_df, saved_data_path='cleaned_data.csv')
df = pd.read_csv('cleaned_data.csv')
df.drop(columns=['rider_id','restaurant_latitude','restaurant_longitude',
                  'delivery_latitude','delivery_longitude','order_day','order_time_hour'],
        inplace=True)

num_col = ['age','ratings','pickup_time_minutes','distance']
nominal_cat_cols = ['type_of_order','type_of_vehicle','festival','city_type',
                    'city_name','weather','order_day_of_week','order_time_of_day']
ordinal_cat_cols = ['distance_type','traffic']
distance_type_order = ['short','very_long','medium','long']
traffic_order = ['low','medium','high','jam']
print(f'Data loaded: {df.shape}')

## 3. Load Best Model Metadata

In [ ]:
with open('best_models_meta.json') as f:
    meta = json.load(f)
m1, m2 = meta['best_model_1'], meta['best_model_2']
print(f"Best Model 1: {m1['name']} ({m1['dataset']})  Exp3 RMSE={m1['rmse']}")
print(f"Best Model 2: {m2['name']} ({m2['dataset']})  Exp3 RMSE={m2['rmse']}")

assert m1['dataset'] == 'Dataset_A' and m2['dataset'] == 'Dataset_A', \
    "Stacking assumes both base models share Dataset_A; adjust split logic otherwise."

df_use = df.dropna().copy()
X = df_use.drop(columns='time_taken')
y = df_use['time_taken']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train:{X_train.shape}  Test:{X_test.shape}')

## 4. Load Tuned Base Pipelines from Exp 4 & Exp 5

(Unchanged — same objects, same hyperparameters)

In [ ]:
best_pipeline_1 = joblib.load('best_model_1_tuned.pkl')   # LightGBM, full preprocessing+model pipeline
best_pipeline_2 = joblib.load('best_model_2_tuned.pkl')   # Random Forest, full preprocessing+model pipeline
pt = joblib.load('pt_model_1.pkl')                        # PowerTransformer fit on Dataset_A target (same for both)

y_train_pt = np.asarray(pt.transform(y_train.values.reshape(-1,1))).reshape(-1)

## 5. Build the Stacking Ensemble

- **Base learners**: best_model_1 (LightGBM) + best_model_2 (Random Forest), unchanged
- **Meta learner**: Linear Regression

In [ ]:
stack_model = StackingRegressor(
    estimators=[
        (m1['name'].lower().replace(' ', '_'), best_pipeline_1),
        (m2['name'].lower().replace(' ', '_'), best_pipeline_2),
    ],
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

stack_model.fit(X_train, y_train_pt)
print('Stacking ensemble trained.')

## 6. Evaluate on Test Set

In [ ]:
def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((np.array(y_true) - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
    return {'RMSE': round(rmse,4), 'MAE': round(mae,4), 'R2': round(r2,4), 'MAPE': round(mape,4)}

y_pred_pt = stack_model.predict(X_test)
y_pred_stack = pt.inverse_transform(y_pred_pt.reshape(-1,1)).ravel()
stack_metrics = compute_metrics(y_test.values, y_pred_stack)
print('Stacking Ensemble — Test Results:', stack_metrics)

# Individual base model test performance for comparison
y_pred_1_pt = best_pipeline_1.predict(X_test)
y_pred_1 = pt.inverse_transform(y_pred_1_pt.reshape(-1,1)).ravel()
metrics_1 = compute_metrics(y_test.values, y_pred_1)

y_pred_2_pt = best_pipeline_2.predict(X_test)
y_pred_2 = pt.inverse_transform(y_pred_2_pt.reshape(-1,1)).ravel()
metrics_2 = compute_metrics(y_test.values, y_pred_2)

comparison_df = pd.DataFrame([
    {'Model': m1['name'], **metrics_1},
    {'Model': m2['name'], **metrics_2},
    {'Model': 'Stacking (LR meta)', **stack_metrics},
]).sort_values('RMSE').reset_index(drop=True)
print(comparison_df.to_string(index=False))
comparison_df.to_csv('exp6_stacking_comparison.csv', index=False)

## 7. Meta-learner Weights (Interpretability)

In [ ]:
final_lr = stack_model.final_estimator_
print('Meta-model (Linear Regression) coefficients:', final_lr.coef_, 'intercept:', final_lr.intercept_)

## 8. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
fig.suptitle('Stacking Ensemble — Predictions', fontsize=13, fontweight='bold')
axes[0].scatter(y_test, y_pred_stack, alpha=0.35, s=10, color='seagreen')
mn, mx = min(y_test.min(), y_pred_stack.min()), max(y_test.max(), y_pred_stack.max())
axes[0].plot([mn,mx],[mn,mx],'r--', lw=2)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted'); axes[0].set_title('Actual vs Predicted')
axes[0].grid(alpha=0.3)

resid = y_test.values - y_pred_stack
axes[1].scatter(y_pred_stack, resid, alpha=0.35, s=10, color='coral')
axes[1].axhline(0, color='black', lw=2, ls='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual'); axes[1].set_title('Residual Plot')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('exp6_stacking_pred_residual.png', dpi=150, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8,5))
sns.barplot(data=comparison_df, x='Model', y='RMSE', palette='viridis')
plt.title('RMSE: Base Models vs Stacking Ensemble', fontweight='bold')
plt.ylabel('RMSE (minutes)'); plt.tight_layout()
plt.savefig('exp6_rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plots saved.')

## 9. Save the Stacking Pipeline + Deployment Bundle

In [ ]:
joblib.dump(stack_model, 'stacking_model.pkl')

from clean_data_utils import DeliveryTimePredictor

drop_cols = ['rider_id','restaurant_latitude','restaurant_longitude',
             'delivery_latitude','delivery_longitude','order_day','order_time_hour']

predictor = DeliveryTimePredictor(stack_model, pt, drop_cols)

# sanity check: predict on a few raw rows
sample_raw = raw_df.sample(3, random_state=1).reset_index(drop=True)
sample_pred = predictor.predict(sample_raw)
print('Sample raw-row predictions (minutes):', sample_pred)

joblib.dump(predictor, 'delivery_time_predictor.pkl')

print('='*55)
print('EXPERIMENT 6 COMPLETE')
print('='*55)
print(f'Stacking RMSE: {stack_metrics["RMSE"]}  R2: {stack_metrics["R2"]}')
print('Saved: stacking_model.pkl, delivery_time_predictor.pkl, exp6_stacking_comparison.csv')
